In [16]:
import os

from langchain_core.messages import SystemMessage, HumanMessage
from pyowm import OWM
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_ollama import OllamaLLM, ChatOllama
from os import environ

load_dotenv(verbose=True)

owm = OWM(environ.get("OWM_API_KEY"))
mgr = owm.weather_manager()

In [17]:
from langchain_core.tools import tool


@tool(response_format="content")
def get_current_weather(place: str) -> dict:
    """
    Returns the current weather for a place.

    Parameters:
         :param place: The place to get the weather for, in a <Place>,<Country Code> format.

    Returns:
        Dictionary of weather data for the place given.

    Example:
    >>> get_current_weather(place='London,GB')
    >>> {"place": "Athens,GR", "status": "few clouds", "temperature_c": 22.96, "feels_like_c": 23.24, "temp_min_c": 22.22, "temp_max_c": 23.97, "humidity_pct": 74, "pressure_hpa": 1018, "wind": {"speed": 5.66, "deg": 160}, "clouds_pct": 20, "rain_mm": {}, "snow_mm": {}, "sunrise_iso": "2025-11-09 04:59:19+00:00", "sunset_iso": "2025-11-09 15:18:41+00:00", "observed_at_iso": "2025-11-09 12:46:02+00:00", "uv_index": null, "visibility_m": 10000, "weather_icon": "02d"}
    """

    w = mgr.weather_at_place(place).weather
    return {
            "place": place,
            "status": w.detailed_status,                           # e.g. 'clear sky'
            "temperature_c": w.temperature("celsius")["temp"],     # current temp (°C)
            "feels_like_c": w.temperature("celsius")["feels_like"],
            "temp_min_c": w.temperature("celsius").get("temp_min"),
            "temp_max_c": w.temperature("celsius").get("temp_max"),
            "humidity_pct": w.humidity,
            "pressure_hpa": w.pressure.get("press"),
            "wind": w.wind(),                                       # {'speed': .., 'deg': .., 'gust': ..}
            "clouds_pct": w.clouds,
            "rain_mm": w.rain,                                      # dict (last 1h/3h) if present
            "snow_mm": w.snow,                                      # dict (last 1h/3h) if present
            "sunrise_iso": w.sunrise_time(timeformat="iso"),
            "sunset_iso": w.sunset_time(timeformat="iso"),
            "observed_at_iso": w.reference_time(timeformat="iso"),
            "uv_index": w.uvi,                                      # may be None depending on API plan
            "visibility_m": w.visibility_distance,
            "weather_icon": w.weather_icon_name,                    # e.g. '01d'
        }


In [18]:
system_prompt = """
You are a helpful weather assistant. Your job is to help the user with their weather requests.

You have the get_current_weather tool at your disposal, which fetches the current weather.
"""

In [19]:
model = ChatOllama(model="gpt-oss:20b")

agent = create_agent(
    model=model,
    tools=[get_current_weather],
    system_prompt=system_prompt
)

In [20]:
request = "Please tell me the current weather in Athens, Greece. What do you think I should wear?"

In [21]:
messages = agent.invoke({
    "messages": [
        HumanMessage(request)
    ]
})["messages"]

[m.pretty_print() for m in messages]

================================ Human Message =================================

Please tell me the current weather in Athens, Greece. What do you think I should wear?
================================== Ai Message ==================================
Tool Calls:
  get_current_weather (c743f0e2-d680-4fe5-98c8-c064bba73da2)
 Call ID: c743f0e2-d680-4fe5-98c8-c064bba73da2
  Args:
    place: Athens,GR
================================= Tool Message =================================
Name: get_current_weather

{"place": "Athens,GR", "status": "scattered clouds", "temperature_c": 16.85, "feels_like_c": 16.11, "temp_min_c": 15.97, "temp_max_c": 17.98, "humidity_pct": 58, "pressure_hpa": 1008, "wind": {"speed": 4.92, "deg": 242, "gust": 6.26}, "clouds_pct": 40, "rain_mm": {}, "snow_mm": {}, "sunrise_iso": "2025-11-29 05:20:31+00:00", "sunset_iso": "2025-11-29 15:06:46+00:00", "observed_at_iso": "2025-11-29 11:29:59+00:00", "uv_index": null, "visibility_m": 10000, "weather_icon": "03d"}
===========

[None, None, None, None]